# SWT-CP Free-Rider Experiment Notebook

Use this notebook to configure, run, and inspect federated learning free-rider experiments. The actual simulator code stays in the `.py` files; this notebook just calls it in a more comfortable way.

## 1. Imports

Run this first. If imports fail, install the packages from `requirements.txt` in the same environment used by Jupyter.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

from config import ExperimentConfig
from train import run_experiment

plt.style.use("default")

## 2. Choose Parameters

This is the main cell you edit. Common choices:

- `dataset`: `"mnist"` or `"cifar10"`
- `distribution`: `"iid"` or `"noniid"`
- `attack_type`: `"FR1"`, `"FR2"`, `"FR3"`, or `"FR4"`
- `free_rider_pct`: `0.1`, `0.2`, or `0.4`
- `device`: `"auto"`, `"cpu"`, or `"cuda"`

In [ ]:
config = ExperimentConfig(
    dataset="mnist",
    distribution="iid",
    dirichlet_alpha=0.5,
    num_clients=100,
    num_rounds=100,
    free_rider_pct=0.10,
    attack_type="FR1",
    local_epochs=3,
    batch_size=32,
    optimizer="sgd",
    learning_rate=0.01,
    momentum=0.9,
    warmup_rounds=10,
    trap_fraction=0.10,
    num_anchors=3,
    seed=42,
    device="auto",
)

config.validate()
config.to_dict()

## 3. Optional Smoke Run

Run this cell first if you only want to check that the pipeline works. It uses a tiny run and writes outputs to `results/notebook_smoke/`.

In [ ]:
smoke_config = ExperimentConfig(
    dataset="mnist",
    distribution="iid",
    num_clients=4,
    num_rounds=2,
    warmup_rounds=1,
    local_epochs=1,
    batch_size=16,
    trap_fraction=0.5,
    num_anchors=1,
    attack_type="FR1",
    free_rider_pct=0.25,
    run_name="notebook_smoke",
    device="auto",
)

# Uncomment the next two lines when you want the quick check.
# smoke_run_dir = Path(run_experiment(smoke_config))
# smoke_run_dir

## 4. Run Experiment

This launches the actual experiment using the `config` object from Section 2. Full 100-client runs can take a while, especially on CPU.

In [ ]:
run_dir = Path(run_experiment(config))
run_dir

## 5. Load Outputs

After a run finishes, this loads the CSV and JSON outputs from `run_dir`.

In [ ]:
global_metrics = pd.read_csv(run_dir / "global_metrics.csv")
client_metrics = pd.read_csv(run_dir / "client_metrics.csv")
penalty_tracker = pd.read_csv(run_dir / "penalty_tracker.csv")
trap_matrix = pd.read_csv(run_dir / "trap_matrix.csv")
removals = pd.read_csv(run_dir / "removals.csv")
detection_summary = pd.read_csv(run_dir / "detection_summary.csv")
run_config = json.loads((run_dir / "run_config.json").read_text())

display(detection_summary)
display(global_metrics.tail())
display(removals.head())

## 6. Plot Global Accuracy and Loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(global_metrics["round"], global_metrics["global_accuracy"])
axes[0].set_title("Global Accuracy")
axes[0].set_xlabel("Round")
axes[0].set_ylabel("Accuracy")

axes[1].plot(global_metrics["round"], global_metrics["global_loss"])
axes[1].set_title("Global Loss")
axes[1].set_xlabel("Round")
axes[1].set_ylabel("Loss")

plt.tight_layout()

## 7. Inspect Penalties

This shows the highest-penalty clients at the end of the run. Free-rider labels are only used here for evaluation, after detection has already happened.

In [ ]:
final_round = penalty_tracker["round"].max()
final_penalties = penalty_tracker[penalty_tracker["round"] == final_round].copy()

free_rider_labels = (
    client_metrics[["client_id", "is_free_rider"]]
    .drop_duplicates("client_id")
    .set_index("client_id")
)
final_penalties = final_penalties.join(free_rider_labels, on="client_id")

display(final_penalties.sort_values("cumulative_penalty", ascending=False).head(20))

## 8. Save a Config Snapshot

Use this if you want to preserve the parameters from Section 2 as a JSON config file for CLI runs later.

In [ ]:
config_path = Path("configs/notebook_config.json")
config_path.parent.mkdir(exist_ok=True)
config_path.write_text(json.dumps(config.to_dict(), indent=2), encoding="utf-8")
config_path